# Weed Detection - DeepWeeds YOLOv8n Training

This notebook trains a YOLOv8n model on the DeepWeeds dataset for real-time
weed detection on the AgroBot rover. The model is exported as a quantized
TFLite model for the Coral Edge TPU.

**Hardware requirements:** Google Colab with T4 GPU runtime.

**Dataset:** DeepWeeds - 17,509 images across 9 weed species common in
Australian rangelands.

**Expected output:** `weed_model_quant.tflite` for deployment to `models/`
on the Raspberry Pi.

In [ ]:
# Install dependencies
!pip install -q ultralytics roboflow matplotlib

In [ ]:
import os
import yaml
import shutil
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

print(f"Working directory: {os.getcwd()}")

In [ ]:
# Download DeepWeeds dataset and convert to YOLO format
# Option 1: From Roboflow (pre-formatted YOLO)
from roboflow import Roboflow

# Replace with your Roboflow API key
rf = Roboflow(api_key="YOUR_API_KEY")
project = rf.workspace().project("deepweeds")
dataset = project.version(1).download("yolov8")

DATASET_DIR = dataset.location
print(f"Dataset downloaded to: {DATASET_DIR}")

# List classes
with open(os.path.join(DATASET_DIR, 'data.yaml')) as f:
    data_config = yaml.safe_load(f)
print(f"Classes: {data_config['names']}")
print(f"Number of classes: {data_config['nc']}")

In [ ]:
# Dataset YAML configuration
# If not using Roboflow, create the dataset config manually:
dataset_yaml = {
    'path': DATASET_DIR,
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 9,
    'names': [
        'chinee_apple',
        'lantana',
        'parkinsonia',
        'parthenium',
        'prickly_acacia',
        'rubber_vine',
        'siam_weed',
        'snake_weed',
        'negatives',
    ],
}

yaml_path = os.path.join(DATASET_DIR, 'dataset.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)

print(f"Dataset YAML saved to: {yaml_path}")
print(f"\nDataset structure:")
!find {DATASET_DIR} -maxdepth 2 -type d

In [ ]:
# YOLOv8n model setup (from pretrained COCO weights)
model = YOLO('yolov8n.pt')  # Load pretrained YOLOv8n

print(f"Model: {model.model_name}")
print(f"Parameters: {sum(p.numel() for p in model.model.parameters()):,}")

In [ ]:
# Training with augmentation parameters
results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=15,
    device=0,
    # Augmentation parameters
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    # Training parameters
    lr0=0.01,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    project='weed_detection',
    name='yolov8n_deepweeds',
)

print(f"\nTraining complete!")
print(f"Best model: {results.save_dir}/weights/best.pt")

In [ ]:
# Validation and metrics visualization
best_model = YOLO(f'{results.save_dir}/weights/best.pt')
val_results = best_model.val(data=yaml_path, imgsz=640)

print(f"\nValidation Metrics:")
print(f"  mAP50:    {val_results.box.map50:.4f}")
print(f"  mAP50-95: {val_results.box.map:.4f}")
print(f"  Precision: {val_results.box.mp:.4f}")
print(f"  Recall:    {val_results.box.mr:.4f}")

# Per-class AP
print(f"\nPer-class mAP50:")
for i, name in enumerate(dataset_yaml['names']):
    if i < len(val_results.box.ap50):
        print(f"  {name:20s}: {val_results.box.ap50[i]:.4f}")

# Display training curves
from IPython.display import Image, display
display(Image(filename=f'{results.save_dir}/results.png', width=800))
display(Image(filename=f'{results.save_dir}/confusion_matrix.png', width=600))
display(Image(filename=f'{results.save_dir}/PR_curve.png', width=600))

In [ ]:
# Export to TFLite format
best_model = YOLO(f'{results.save_dir}/weights/best.pt')

# Export to TFLite with INT8 quantization
tflite_path = best_model.export(
    format='tflite',
    imgsz=320,  # Smaller input for Edge TPU efficiency
    int8=True,
)
print(f"TFLite model exported to: {tflite_path}")

# Check model size
model_size = os.path.getsize(tflite_path) / 1024 / 1024
print(f"Model size: {model_size:.2f} MB")

In [ ]:
# Edge TPU compilation
!curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key add -
!echo "deb https://packages.cloud.google.com/apt coral-edgetpu-stable main" | sudo tee /etc/apt/sources.list.d/coral-edgetpu.list
!sudo apt-get update && sudo apt-get install -y edgetpu-compiler

# Compile for Edge TPU
!edgetpu_compiler -s -o output/ {tflite_path}

print("\nEdge TPU compilation complete!")
!ls -la output/

In [ ]:
# Inference demo on sample images
import glob

# Run inference on test images
test_images = glob.glob(os.path.join(DATASET_DIR, 'test', 'images', '*.jpg'))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for idx, img_path in enumerate(test_images):
    results = best_model.predict(img_path, conf=0.25, imgsz=640)
    ax = axes[idx // 3, idx % 3]
    # Plot result image
    result_img = results[0].plot()
    ax.imshow(result_img[..., ::-1])  # BGR to RGB
    ax.set_title(f"Detections: {len(results[0].boxes)}")
    ax.axis('off')

plt.suptitle('Weed Detection - Sample Predictions', fontsize=14)
plt.tight_layout()
plt.show()

# Print detection details for first image
print("\nDetection details (first image):")
for box in results[0].boxes:
    cls_id = int(box.cls)
    conf = float(box.conf)
    name = dataset_yaml['names'][cls_id]
    print(f"  {name}: {conf:.3f}")

In [ ]:
# Copy model to pi/models/ instructions
#
# The exported model should be placed at:
#   models/weed_model_quant_edgetpu.tflite
#
# This is loaded by pi/ai/weed_detection.py:
#   WeedDetector(model_path='models/weed_model_quant_edgetpu.tflite')
#
# Download from Colab:
from google.colab import files

# Find the edgetpu compiled model
edgetpu_model = [f for f in os.listdir('output') if 'edgetpu' in f][0]
files.download(f'output/{edgetpu_model}')

print(f"Downloaded: output/{edgetpu_model}")
print("Rename to 'weed_model_quant_edgetpu.tflite' and place in models/ directory.")